# 02 – Kiểm tra chất lượng dữ liệu trạm KAK (Bước 2)

**Mục tiêu:** Báo cáo đầy đủ các chỉ số chất lượng của dữ liệu địa từ trạm KAK 1 phút TRƯỚC khi thực hiện làm sạch:
- Giá trị thiếu (Missing values & Sentinel 99999.00)
- Mốc thời gian trùng lặp (Duplicate timestamps)
- Mốc thời gian bị gián đoạn (Time gaps)
- Giá trị vượt ngưỡng vật lý (Out-of-range physical bounds)
- Biến động đột biến (Spikes)
- Chuỗi giá trị không đổi kéo dài (Flatlines)

**Nguyên tắc:** KHÔNG chỉnh sửa dữ liệu trong notebook này, chỉ phát hiện và lưu lại nhật ký QC.


In [1]:
# ── Setup môi trường & nạp thư viện ──────────────────────────────────────────
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from common.parsers import load_all_intermagnet
from common.quality_checks import run_all_checks

print('Setup môi trường thành công.')


Setup môi trường thành công.


In [2]:
# ── Nạp dữ liệu thô trạm KAK (chưa qua xử lý) ────────────────────────────────
station_dfs = {}
for station in cfg.STATIONS:
    try:
        df, _ = load_all_intermagnet(station)
        station_dfs[station] = df
        print(f'✅ Trạm {station}: Nạp thành công {len(df):,} dòng dữ liệu thô')
    except FileNotFoundError as e:
        print(f'⚠  {e}')


✅ Trạm KAK: Nạp thành công 1,707,840 dòng dữ liệu thô


In [3]:
# ── Chạy kiểm tra chất lượng (QC) cho dữ liệu KAK ───────────────────────────
qc_results_before = {}  # Lưu kết quả QC trước khi xử lý để phục vụ Bước 4 & 5

for station, df in station_dfs.items():
    result = run_all_checks(df, station=station, label='before')
    qc_results_before[station] = result
    print(f'  → Hoàn thành QC trước xử lý cho trạm {station}')



──────────────────────────────────────────────────
  Kiểm tra chất lượng | KAK | BEFORE
──────────────────────────────────────────────────
  1. Thiếu giá trị (NaN + sentinel)...
column  nan_count  sentinel_count  total_missing  missing_pct
  x_nt          0               0              0          0.0
  y_nt          0               0              0          0.0
  z_nt          0               0              0          0.0
  f_nt          0               0              0          0.0

  2. Trùng timestamp...
     Dòng liên quan: 0 | Trùng: 0

  3. Thiếu timestamp (gap)...
     Kỳ vọng: 1707840 | Thực tế: 1707840 | Thiếu: 0 | Số gap: 0 | Gap dài nhất: 0 phút

  4. Ngoài khoảng vật lý...
column  bound_min  bound_max  below_min  above_max  out_of_range  out_pct
  x_nt      20000      50000          0          0             0      0.0
  y_nt      -5000      10000          0          0             0      0.0
  z_nt      20000      60000          0          0             0      0.0
  f_nt   

In [4]:
# ── Lưu kết quả QC interim (qc_before.pkl & raw_parquet) ──────────────────────
interim_dir = cfg.DATA_INTERIM
interim_dir.mkdir(parents=True, exist_ok=True)

# Lưu kết quả QC dưới dạng pickle để notebook 03 nạp lại nhanh chóng
qc_path = interim_dir / 'qc_before.pkl'
with open(qc_path, 'wb') as f:
    pickle.dump(qc_results_before, f)
print(f'✅ Đã lưu kết quả QC: {qc_path}')

# Lưu DataFrame thô dưới dạng parquet interim
for station, df in station_dfs.items():
    raw_parquet = interim_dir / f'raw_{station}.parquet'
    df.to_parquet(raw_parquet, index=False)
    print(f'✅ Đã lưu raw Parquet interim: {raw_parquet}')


✅ Đã lưu kết quả QC: /home/khang/Code/data-pipeline/geopulse-kak-usgs/data/interim/qc_before.pkl
✅ Đã lưu raw Parquet interim: /home/khang/Code/data-pipeline/geopulse-kak-usgs/data/interim/raw_KAK.parquet


In [5]:
# ── In bảng tóm tắt chỉ số chất lượng dữ liệu KAK ───────────────────────────
print('\n' + '='*65)
print('  TÓM TẮT BƯỚC 2 – KIỂM TRA CHẤT LƯỢNG KAK (TRƯỚC XỬ LÝ)')
print('='*65)

for station, result in qc_results_before.items():
    print(f'\n[{station}]')

    # Missing values
    miss = result.get('missing')
    if miss is not None and not miss.empty:
        total_miss = miss['total_missing'].sum()
        print(f'  Thiếu giá trị : {total_miss:,} điểm')

    # Duplicate timestamps
    dup = result.get('duplicates', {})
    print(f'  Trùng mốc TG  : {dup.get("duplicate_count", 0):,}')

    # Time gaps
    gap = result.get('gaps', {})
    print(f'  Khoảng gián đoạn (Gap): {gap.get("missing_points", 0):,} điểm ({gap.get("n_gaps", 0)} đoạn gap, dài nhất {gap.get("max_gap_minutes", 0)} phút)')

    # Out of range bounds
    bnd = result.get('bounds')
    if bnd is not None and not bnd.empty:
        total_out = bnd['out_of_range'].sum()
        print(f'  Vượt khoảng v.lý : {total_out:,}')

    # Spikes
    spk = result.get('spikes')
    if spk is not None and not spk.empty:
        total_spk = spk['spike_count'].sum()
        print(f'  Biến động Spike  : {total_spk:,}')

    # Flatlines
    flt = result.get('flatlines')
    if flt is not None and not flt.empty:
        total_flt = flt['total_flagged_points'].sum()
        print(f'  Đứng yên Flatline: {total_flt:,} điểm ({flt["n_segments"].sum()} đoạn)')

print('\n→ Hoàn thành Bước 2. Tiếp tục chuyển sang Bước 3 (Xử lý & Làm sạch dữ liệu KAK).')



  TÓM TẮT BƯỚC 2 – KIỂM TRA CHẤT LƯỢNG KAK (TRƯỚC XỬ LÝ)

[KAK]
  Thiếu giá trị : 0 điểm
  Trùng mốc TG  : 0
  Khoảng gián đoạn (Gap): 0 điểm (0 đoạn gap, dài nhất 0 phút)
  Vượt khoảng v.lý : 0
  Biến động Spike  : 31,727
  Đứng yên Flatline: 2,476 điểm (214 đoạn)

→ Hoàn thành Bước 2. Tiếp tục chuyển sang Bước 3 (Xử lý & Làm sạch dữ liệu KAK).
